# 01 · The Bad

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/01_the_bad.ipynb)

**The failure modes that bite every clustering project.**

Clustering promises structure discovery. In practice you get back... *something*. The hard question is whether that something is real — and most of the time, the algorithm won't tell you.

This notebook walks through four failure modes on a concrete dataset: the **top 5,000 movies** by TMDB vote count. K-means appears as a demonstrator in a few of them, but it's not the villain — the failure modes are.

| # | Failure mode | What it looks like |
| - | --- | --- |
| 1 | Clusters from nothing | The algorithm returns clean-looking clusters on data with no real structure. |
| 2 | Instability across runs | Same data, different seed or init — different story. |
| 3 | Raw high-dim embeddings | Distances compress; everything looks equidistant from everything else. |
| 4 | Parameter choices *become* the story | Small knob changes flip cluster counts and interpretations. |

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q sentence-transformers scikit-learn umap-learn hdbscan pandas pyarrow matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## Load the movies

Generated by `scripts/fetch_movies.py` — see the repo [README](../README.md#getting-the-data) for how to pull your own copy.

In [ ]:
movies = pd.read_parquet("../data/movies.parquet")
movies = movies[movies["overview"].str.len() > 30].reset_index(drop=True)
print(f"{len(movies):,} movies with non-trivial overviews")
movies[["title", "release_year", "genres", "vote_count"]].head()

## Failure mode 1 — Clusters from nothing

Shuffle the words inside each movie overview so the semantic structure is destroyed but the surface statistics are preserved. Embed, cluster, interpret. The algorithm will cheerfully return "themes" for text that has no themes.

**The lesson:** clustering algorithms don't validate that clusters exist. They return a partition regardless. If your only evidence for structure is "the algorithm gave me 5 groups," you have no evidence.

In [ ]:
# TODO: word-shuffle overviews, embed, KMeans(n_clusters=8), inspect 'top terms' per cluster

## Failure mode 2 — Instability across runs

Take the real overviews, embed once, then cluster three times with different seeds. Compare membership with adjusted Rand index. If clusters are real, this number is high. If they're not, it's closer to zero.

**The lesson:** if your downstream decisions depend on specific cluster identities, you need to check that those identities are stable under perturbation.

In [ ]:
# TODO: embed overviews once, 3x KMeans with different random_state, pairwise ARI

## Failure mode 3 — Raw high-dimensional embeddings

Sentence-transformer embeddings are 384-D. At that dimensionality pairwise distances compress: nearest and farthest neighbors sit in a narrow band. Plot the distance distribution and you'll see the problem.

**The lesson:** you almost always want to reduce before you cluster. This is the teaser for [`02_the_good.ipynb`](02_the_good.ipynb).

In [ ]:
# TODO: histogram of pairwise distances in raw 384-D vs. UMAP(5-D)

## Failure mode 4 — Parameter choices become the story

Sweep `k` for k-means on the same embeddings and watch the cluster count — and the narrative you'd tell about it — flip. If the story changes with a knob turn, the story isn't about the data.

**The lesson:** defensible clustering requires sensitivity analysis, not a single "best" config.

In [ ]:
# TODO: k in {5, 10, 20, 40}, compare cluster compositions for a fixed sample movie

## What "The Good" does differently

- Reduces dimensionality first so geometry is interpretable.
- Uses a density-based clusterer that can say *"this movie isn't in any cluster."*
- Treats parameters as ranges to sweep, not oracles.
- Labels clusters with interpretable representations so humans can sanity-check.

Next: [`02_the_good.ipynb`](02_the_good.ipynb).